In [1]:
import os, sys, json
import pandas as pd
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

PROJECT_ROOT  = r'C:\Users\chiku\OneDrive\Documents\ecom_dynamic_pricing_optimization'
PROCESSED_DIR = os.path.join(PROJECT_ROOT, 'data', 'processed')
SRC_DIR       = os.path.join(PROJECT_ROOT, 'src')
sys.path.insert(0, SRC_DIR)

# Load data
train = pd.read_parquet(os.path.join(PROCESSED_DIR, 'train_with_cate.parquet'))

with open(os.path.join(PROCESSED_DIR, 'col_definitions.json')) as f:
    cols = json.load(f)

Y_col  = cols['Y_col']
T_col  = cols['T_col']
W_cols = cols['W_cols']
X_cols = cols['X_cols']

print(f'Loaded: {train.shape}')
print(f'Y={Y_col}, T={T_col}')
print(f'Torch available: ', end='')

import torch
print(torch.__version__)
print(f'Device: {"GPU" if torch.cuda.is_available() else "CPU"}')

Loaded: (500000, 21)
Y=log_units, T=log_price
Torch available: 2.5.1+cpu
Device: CPU


In [2]:
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_percentage_error, r2_score
import time

# ─── Prepare data ─────────────────────────────────────────────────────────────
feature_cols = [T_col] + W_cols
X = train[feature_cols].fillna(0).values
y = train[Y_col].values

# Scale features — critical for neural networks
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, y, test_size=0.2, random_state=42
)

# Convert to tensors
X_train_t = torch.FloatTensor(X_train)
y_train_t = torch.FloatTensor(y_train).unsqueeze(1)
X_test_t  = torch.FloatTensor(X_test)
y_test_t  = torch.FloatTensor(y_test).unsqueeze(1)

# DataLoader — batch training
train_dataset = TensorDataset(X_train_t, y_train_t)
train_loader  = DataLoader(train_dataset, batch_size=2048, shuffle=True)

print(f'Train: {X_train.shape}, Test: {X_test.shape}')
print(f'Input features: {len(feature_cols)} → {feature_cols}')

Train: (400000, 10), Test: (100000, 10)
Input features: 10 → ['log_price', 'price_vs_category', 'log_popularity', 'is_weekend', 'is_morning', 'is_afternoon', 'is_evening', 'is_frequent_shopper', 'days_since_prior_order', 'department_code']


In [3]:
# ─── Deep & Cross Network Architecture ───────────────────────────────────────
# Economics motivation: DCN captures both:
#   - Cross network: explicit feature interactions (price × department,
#                    price × popularity) — like interaction terms in econometrics
#   - Deep network:  nonlinear demand patterns (diminishing sensitivity,
#                    threshold effects)
# Together they approximate a flexible demand function Q = f(P, W)

class CrossLayer(nn.Module):
    """
    Cross network layer: x_out = x_0 * x^T * w + b + x
    Captures explicit bounded-degree feature interactions.
    """
    def __init__(self, input_dim):
        super().__init__()
        self.w = nn.Linear(input_dim, 1, bias=False)
        self.b = nn.Parameter(torch.zeros(input_dim))

    def forward(self, x0, x):
        cross = self.w(x) * x0 + self.b
        return cross + x


class DCN(nn.Module):
    """
    Deep & Cross Network for demand prediction.
    
    Architecture:
        Input (10 features)
            ├── Cross Network (2 layers) — feature interactions
            └── Deep Network (3 layers) — nonlinear patterns
                    ↓
              Concatenate
                    ↓
              Output layer → predicted log(units)
    """
    def __init__(self, input_dim, cross_layers=2, deep_units=64, deep_layers=3, dropout=0.2):
        super().__init__()
        self.input_dim = input_dim

        # Cross network
        self.cross_layers = nn.ModuleList(
            [CrossLayer(input_dim) for _ in range(cross_layers)]
        )

        # Deep network
        deep = []
        in_dim = input_dim
        for _ in range(deep_layers):
            deep += [
                nn.Linear(in_dim, deep_units),
                nn.ReLU(),
                nn.Dropout(dropout),
            ]
            in_dim = deep_units
        self.deep_network = nn.Sequential(*deep)

        # Final output layer
        self.output = nn.Linear(input_dim + deep_units, 1)

    def forward(self, x):
        # Cross network
        x0 = x
        x_cross = x
        for layer in self.cross_layers:
            x_cross = layer(x0, x_cross)

        # Deep network
        x_deep = self.deep_network(x)

        # Concatenate and output
        combined = torch.cat([x_cross, x_deep], dim=1)
        return self.output(combined)


# Instantiate model
input_dim = X_train.shape[1]
model = DCN(input_dim=input_dim, cross_layers=2, deep_units=64,
            deep_layers=3, dropout=0.2)

# Count parameters
n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'DCN Architecture:')
print(f'  Input dim:      {input_dim}')
print(f'  Cross layers:   2')
print(f'  Deep layers:    3 × 64 units')
print(f'  Total params:   {n_params:,}')
print(f'\nModel structure:')
print(model)

DCN Architecture:
  Input dim:      10
  Cross layers:   2
  Deep layers:    3 × 64 units
  Total params:   9,139

Model structure:
DCN(
  (cross_layers): ModuleList(
    (0-1): 2 x CrossLayer(
      (w): Linear(in_features=10, out_features=1, bias=False)
    )
  )
  (deep_network): Sequential(
    (0): Linear(in_features=10, out_features=64, bias=True)
    (1): ReLU()
    (2): Dropout(p=0.2, inplace=False)
    (3): Linear(in_features=64, out_features=64, bias=True)
    (4): ReLU()
    (5): Dropout(p=0.2, inplace=False)
    (6): Linear(in_features=64, out_features=64, bias=True)
    (7): ReLU()
    (8): Dropout(p=0.2, inplace=False)
  )
  (output): Linear(in_features=74, out_features=1, bias=True)
)


In [4]:
# ─── Training ─────────────────────────────────────────────────────────────────
criterion = nn.MSELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001, weight_decay=1e-5)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, patience=3, factor=0.5, verbose=False
)

EPOCHS = 20
train_losses = []
val_losses   = []

print(f'Training DCN for {EPOCHS} epochs...')
print(f'Batch size: 2048, Optimizer: Adam, LR: 0.001')
print('-' * 45)

start = time.time()

for epoch in range(EPOCHS):
    # ── Training phase ──
    model.train()
    epoch_loss = 0
    for X_batch, y_batch in train_loader:
        optimizer.zero_grad()
        y_pred = model(X_batch)
        loss = criterion(y_pred, y_batch)
        loss.backward()
        optimizer.step()
        epoch_loss += loss.item()

    avg_train_loss = epoch_loss / len(train_loader)
    train_losses.append(avg_train_loss)

    # ── Validation phase ──
    model.eval()
    with torch.no_grad():
        val_pred = model(X_test_t)
        val_loss = criterion(val_pred, y_test_t).item()
    val_losses.append(val_loss)

    scheduler.step(val_loss)

    if (epoch + 1) % 5 == 0:
        print(f'Epoch {epoch+1:>3}/{EPOCHS} | '
              f'Train Loss: {avg_train_loss:.4f} | '
              f'Val Loss: {val_loss:.4f}')

elapsed = time.time() - start
print(f'\n✅ Training complete in {elapsed:.1f} seconds')

# ─── Evaluation ───────────────────────────────────────────────────────────────
model.eval()
with torch.no_grad():
    y_pred_np = model(X_test_t).numpy().flatten()

dcn_mape = mean_absolute_percentage_error(y_test, y_pred_np)
dcn_r2   = r2_score(y_test, y_pred_np)

print(f'\nDCN Test Results:')
print(f'  MAPE:  {dcn_mape:.4f}')
print(f'  R²:    {dcn_r2:.4f}')

# ─── Learning curve plot ──────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(range(1, EPOCHS+1), train_losses, label='Train Loss', color='#2563EB', lw=2)
ax.plot(range(1, EPOCHS+1), val_losses,   label='Val Loss',   color='#EA580C', lw=2)
ax.set_xlabel('Epoch')
ax.set_ylabel('MSE Loss')
ax.set_title('DCN Learning Curve')
ax.legend()
plt.tight_layout()
plt.savefig(os.path.join(PROCESSED_DIR, 'dcn_learning_curve.png'), dpi=150, bbox_inches='tight')
plt.show()
print('Learning curve saved ✅')

Training DCN for 20 epochs...
Batch size: 2048, Optimizer: Adam, LR: 0.001
---------------------------------------------
Epoch   5/20 | Train Loss: 0.5644 | Val Loss: 0.5334
Epoch  10/20 | Train Loss: 0.5455 | Val Loss: 0.5331
Epoch  15/20 | Train Loss: 0.5344 | Val Loss: 0.5305
Epoch  20/20 | Train Loss: 0.5319 | Val Loss: 0.5298

✅ Training complete in 405.5 seconds

DCN Test Results:
  MAPE:  0.4243
  R²:    0.0319
Learning curve saved ✅


In [ ]:
import joblib

# ─── Save DCN model and scaler ────────────────────────────────────────────────
models_dir = os.path.join(PROJECT_ROOT, 'models')
os.makedirs(models_dir, exist_ok=True)

torch.save(model.state_dict(), os.path.join(models_dir, 'dcn_model.pt'))
joblib.dump(scaler, os.path.join(models_dir, 'dcn_scaler.pkl'))

# ─── Load baseline results for comparison ─────────────────────────────────────
with open(os.path.join(PROCESSED_DIR, 'baseline_results.json')) as f:
    baseline = json.load(f)

# ─── Model comparison table ───────────────────────────────────────────────────
print('=' * 55)
print('MODEL COMPARISON — DEMAND PREDICTION')
print('=' * 55)
print(f'{"Model":<25} {"MAPE":>8} {"R²":>8} {"Notes"}')
print('-' * 55)
print(f'{"Ridge Regression":<25} {baseline["ridge_mape"]:>8.4f} {baseline["ridge_r2"]:>8.4f}  ← linear baseline')
print(f'{"DCN (Deep & Cross)":<25} {dcn_mape:>8.4f} {dcn_r2:>8.4f}  ← neural network')
print('-' * 55)
print(f'\nDCN improvement over Ridge:')
print(f'  MAPE: {(baseline["ridge_mape"] - dcn_mape) / baseline["ridge_mape"]:+.1%}')
print(f'  R²:   {dcn_r2 - baseline["ridge_r2"]:+.4f}')

# ─── Save DCN results ─────────────────────────────────────────────────────────
dcn_results = {
    'dcn_mape': round(dcn_mape, 6),
    'dcn_r2': round(dcn_r2, 6),
    'training_time_seconds': round(elapsed, 1),
    'n_params': n_params,
    'epochs': EPOCHS,
}
with open(os.path.join(PROCESSED_DIR, 'dcn_results.json'), 'w') as f:
    json.dump(dcn_results, f, indent=2)

print(f'\n✅ DCN model saved     → models/dcn_model.pt')
print(f'✅ DCN scaler saved    → models/dcn_scaler.pkl')
print(f'✅ DCN results saved   → data/processed/dcn_results.json')
print(f'\nNotebook 05 complete — ready for notebook 06 (hyperparameter tuning)')

MODEL COMPARISON — DEMAND PREDICTION
Model                         MAPE       R² Notes
-------------------------------------------------------
Ridge Regression            0.4292   0.0176  ← linear baseline
DCN (Deep & Cross)          0.4243   0.0319  ← neural network
-------------------------------------------------------

DCN improvement over Ridge:
  MAPE: +1.1%
  R²:   +0.0143

✅ DCN model saved     → models/dcn_model.pt
✅ DCN scaler saved    → models/dcn_scaler.pkl
✅ DCN results saved   → data/processed/dcn_results.json

Notebook 05 complete — ready for notebook 06 (hyperparameter tuning)


: 